### Imports

In [0]:
# ============================================================================
# SMART MANUFACTURING INTELLIGENCE PLATFORM (SMIP)
# Gold Layer
#
# Notebook : 02_quality_summary
# Layer    : Gold
#
# Description
# ----------------------------------------------------------------------------
# Creates the business quality summary table from Silver facts and dimensions.
#
# Grain
# ----------------------------------------------------------------------------
# Production Date
# +
# Product
# +
# Planned Shift
# ============================================================================

from pyspark.sql.functions import (
    col,
    count,
    countDistinct,
    sum,
    avg,
    round,
    when,
    year,
    quarter,
    month,
    weekofyear,
    to_date,
    date_format
)

from framework.core.session import spark

from framework.core.configuration import (
    SILVER_LAYER,
    GOLD_LAYER
)

from framework.core.logger import (
    banner,
    info,
    success
)

from framework.io.delta import write_delta

In [0]:
banner("Gold Layer - Quality Summary")

### Reading Silver Tables

In [0]:
quality = (

    spark.table(f"{SILVER_LAYER}.fact_quality")

    .select(

        "quality_key",

        "product_key",

        "execution_id",

        "serial_number",

        "product_code",

        "test_name",

        "target_value",

        "measured_value",

        "result",

        "start_time"

    )

)

production = (

    spark.table(f"{SILVER_LAYER}.fact_production")

    .select(

        "execution_id",

        "planned_shift"

    )

)

products = (

    spark.table(f"{SILVER_LAYER}.dim_products")

    .select(

        "product_key",

        "product_name",

        "family",

        "rated_voltage_kv"

    )

)

info("Silver tables loaded successfully.")

### Join

In [0]:
df = (

    quality.alias("q")

    .join(

        production.alias("p"),

        "execution_id",

        "left"

    )

    .join(

        products.alias("d"),

        "product_key",

        "left"

    )

    .select(

        col("q.quality_key"),

        col("q.product_key"),

        col("q.execution_id"),

        col("q.serial_number"),

        col("q.product_code"),

        col("d.product_name"),

        col("d.family"),

        col("d.rated_voltage_kv"),

        col("p.planned_shift"),

        col("q.test_name"),

        col("q.target_value"),

        col("q.measured_value"),

        col("q.result"),

        col("q.start_time")

    )

)

### Calendar Columns

In [0]:
df = (

    df

    .withColumn(

        "production_date",

        to_date("start_time")

    )

    .withColumn(

        "production_year",

        year("start_time")

    )

    .withColumn(

        "production_quarter",

        quarter("start_time")

    )

    .withColumn(

        "production_month",

        month("start_time")

    )

    .withColumn(

        "production_week",

        weekofyear("start_time")

    )

    .withColumn(

        "production_day",

        date_format(

            "start_time",

            "EEEE"

        )

    )

)

### Business Aggregation

In [0]:
quality_summary = (

    df

    .groupBy(

        "production_date",

        "production_year",

        "production_quarter",

        "production_month",

        "production_week",

        "production_day",

        "planned_shift",

        "product_key",

        "product_code",

        "product_name",

        "family",

        "rated_voltage_kv"

    )

    .agg(

        count("*").alias("tests_performed"),

        countDistinct("serial_number").alias("units_tested"),

        sum(

            when(

                col("result") == "PASS",

                1

            ).otherwise(0)

        ).alias("passed_tests"),

        sum(

            when(

                col("result") == "FAIL",

                1

            ).otherwise(0)

        ).alias("failed_tests"),

        round(

            avg("measured_value"),

            2

        ).alias("average_measured_value")

    )

)

### Derived KPIs

In [0]:
quality_summary = (

    quality_summary

    .withColumn(

        "pass_rate",

        round(

            (col("passed_tests") * 100.0)

            /

            col("tests_performed"),

            2

        )

    )

    .withColumn(

        "fail_rate",

        round(

            (col("failed_tests") * 100.0)

            /

            col("tests_performed"),

            2

        )

    )

)

### Write Gold

In [0]:
write_delta(

    quality_summary,

    f"{GOLD_LAYER}.quality_summary"

)

success("gold.quality_summary created successfully.")

### Validation

In [0]:
# Preview
display(quality_summary)

#Row Count
display(

    spark.sql(f"""

    SELECT

        COUNT(*) rows

    FROM {GOLD_LAYER}.quality_summary

    """)

)

#Daily Pass Rate

display(

    spark.sql(f"""

    SELECT

        production_date,

        ROUND(

            AVG(pass_rate),

            2

        ) AS average_pass_rate

    FROM {GOLD_LAYER}.quality_summary

    GROUP BY production_date

    ORDER BY production_date

    """)

)

#Shift Quality

display(

    spark.sql(f"""

    SELECT

        planned_shift,

        ROUND(

            AVG(pass_rate),

            2

        ) AS pass_rate

    FROM {GOLD_LAYER}.quality_summary

    GROUP BY planned_shift

    ORDER BY pass_rate DESC

    """)

)

#Product Quality

display(

    spark.sql(f"""

    SELECT

        product_name,

        ROUND(

            AVG(pass_rate),

            2

        ) AS pass_rate,

        SUM(tests_performed) AS tests

    FROM {GOLD_LAYER}.quality_summary

    GROUP BY product_name

    ORDER BY pass_rate DESC

    """)

)